# Foundations of Data Science
## Tutorial 9 — VC dimension, capacity, and why deep learning breaks classical generalization theory

Reproducing the central experiments of a landmark paper:

> **Zhang, Bengio, Hardt, Recht, Vinyals (2017),** *Understanding deep learning requires rethinking generalization.* [arXiv:1611.03530](https://arxiv.org/pdf/1611.03530)

### The puzzle

Classical statistical learning theory explains generalization through **capacity control**. The story goes:
a model class that is too rich can fit *anything*, including noise, so it should not generalize — unless we
restrict its capacity (few parameters, large margins, strong regularization). The **VC dimension** is the
classical way to measure that capacity, and it plugs into a bound of the form

$$ \underbrace{R(f)}_{\text{test error}} \;\le\; \underbrace{\hat R(f)}_{\text{train error}} \;+\; O\!\left(\sqrt{\frac{d_{\mathrm{VC}}}{n}}\right). $$

Modern deep networks have **far more parameters than training examples**, so their VC dimension is astronomically
large and this bound is *vacuous* — it predicts test error could be anything. And yet they generalize extremely well.

Zhang et al. ran a deceptively simple experiment that made the contradiction unavoidable:
**state-of-the-art networks can fit a training set of completely random labels to zero error.**
Their effective capacity is therefore large enough to memorize the data outright — so whatever makes them
generalize on *real* data cannot be explained by capacity alone.

Today we reproduce this on the 8×8 digits dataset with a from-scratch NumPy MLP, and connect it back to VC dimension.

**Plan**
- **Part 0** — VC dimension & shattering, made concrete with linear classifiers.
- **Part 1** — The randomization test: fit *true* vs *random* labels.
- **Part 2** — The label-corruption sweep (reproducing Figure 1 of the paper).
- **Part 3** — Random pixels and Gaussian inputs: destroying structure entirely.
- **Part 4** — Why the VC bound goes vacuous, and finite-sample expressivity (the paper's Theorem 1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from itertools import product

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (9, 4)

## Part 0 — VC dimension and shattering

A hypothesis class $\mathcal{H}$ (e.g. *all linear classifiers in the plane*) **shatters** a set of points if,
for **every** possible labeling of those points with $\{-1,+1\}$, some $h \in \mathcal{H}$ realizes it.

The **VC dimension** $d_{\mathrm{VC}}(\mathcal{H})$ is the size of the *largest* set of points that $\mathcal{H}$ can shatter.
It measures capacity in a label-counting sense: how complicated a pattern of $\pm 1$ the class can carve out.

**Classic fact:** linear classifiers (a separating line + bias) in $\mathbb{R}^2$ have $d_{\mathrm{VC}} = 3$.
Three points in general position can be split in all $2^3 = 8$ ways; but **no** set of 4 points can be split in all
$2^4 = 16$ ways (the XOR labeling defeats every line). Let's see both facts.

In [ ]:
def linearly_separable(points, labels):
    """Can a line (w·x + b) realize this ±1 labeling? Solve a tiny soft-margin SVM by SGD."""
    X = np.asarray(points, float); y = np.asarray(labels, float)
    if len(np.unique(y)) == 1:
        return True  # all one class is trivially realizable
    w = np.zeros(2); b = 0.0
    rg = np.random.default_rng(0)
    for _ in range(5000):
        i = rg.integers(len(X))
        if y[i] * (X[i] @ w + b) < 1:
            w += 0.01 * (y[i] * X[i] - 0.001 * w); b += 0.01 * y[i]
    margins = y * (X @ w + b)
    return bool(np.all(margins > 0))   # all points on the correct side?

def shatters(points):
    """Try every ±1 labeling; return the fraction that a line can realize."""
    pts = np.asarray(points, float); k = len(pts)
    realizable = sum(linearly_separable(pts, lab)
                     for lab in product([-1, 1], repeat=k))
    return realizable, 2**k

In [ ]:
# 3 points in general position (a triangle)
tri = [[0, 0], [1, 0], [0.5, 1]]
r3, t3 = shatters(tri)
print(f"3 points: {r3}/{t3} labelings realizable by a line  -> shattered: {r3 == t3}")

# 4 points (a square) — the XOR labeling will break it
sq = [[0, 0], [1, 0], [1, 1], [0, 1]]
r4, t4 = shatters(sq)
print(f"4 points: {r4}/{t4} labelings realizable by a line  -> shattered: {r4 == t4}")

In [ ]:
# Visualize: the 8 labelings of 3 points (all separable) and the killer XOR labeling of 4 points
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

tri = np.array(tri)
ax = axes[0]
for lab in product([-1, 1], repeat=3):
    pass
# show one nontrivial labeling of the triangle that IS separable
lab3 = np.array([1, -1, 1])
ax.scatter(*tri[lab3 == 1].T, c="crimson", s=200, label="+1")
ax.scatter(*tri[lab3 == -1].T, c="royalblue", s=200, marker="s", label="-1")
ax.set_title("3 points: every labeling is\nlinearly separable  (VC = 3)")
ax.legend(); ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)

ax = axes[1]
sq = np.array(sq)
labxor = np.array([1, -1, 1, -1])  # opposite corners share a class = XOR
ax.scatter(*sq[labxor == 1].T, c="crimson", s=200, label="+1")
ax.scatter(*sq[labxor == -1].T, c="royalblue", s=200, marker="s", label="-1")
ax.set_title("4 points, XOR labeling:\nno single line separates them")
ax.legend(); ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
plt.tight_layout(); plt.show()

**The takeaway from Part 0.** VC dimension is exactly *"how many points can this class label arbitrarily?"*
For a linear classifier in $d$ dimensions it is $d+1$ — small, and the generalization bound above is meaningful.

Now hold that definition in mind. The Zhang et al. experiment asks the empirical question hiding inside it:
**how many points can a deep network label arbitrarily?** If the answer is "all of them, even with random labels,"
then its effective VC dimension is at least $n$, the bound becomes $\hat R + O(\sqrt{n/n}) = \hat R + O(1)$, and it
tells us nothing.

## A from-scratch MLP (our "deep network")

We reuse the two-layer ReLU network from Tutorial 7, but make the hidden layer **wide** (512 units) so it has
plenty of capacity. With 64 inputs that is $64\cdot512 + 512\cdot10 \approx 38{,}000$ parameters — more than the
$\sim$1,400 training images. By the classical worry, it *should* overfit catastrophically. We'll see what actually happens.

The only addition versus Tutorial 7 is an optional **weight-decay** term (L2 regularization), which we leave at 0 for now
and switch on in an exercise.

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def one_hot(y, n_classes=10):
    Y = np.zeros((len(y), n_classes)); Y[np.arange(len(y)), y] = 1
    return Y

class MLP:
    def __init__(self, in_dim, hidden, out_dim, rng):
        self.W1 = rng.standard_normal((in_dim, hidden)) * np.sqrt(2 / in_dim)
        self.b1 = np.zeros(hidden)
        self.W2 = rng.standard_normal((hidden, out_dim)) * np.sqrt(2 / hidden)
        self.b2 = np.zeros(out_dim)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.h = np.maximum(0, self.z1)          # ReLU
        self.z2 = self.h @ self.W2 + self.b2
        return softmax(self.z2)

    def step(self, X, Y, lr, weight_decay=0.0):
        p = self.forward(X); n = len(X)
        dz2 = (p - Y) / n
        dW2 = self.h.T @ dz2 + weight_decay * self.W2; db2 = dz2.sum(0)
        dz1 = (dz2 @ self.W2.T) * (self.z1 > 0)
        dW1 = X.T @ dz1 + weight_decay * self.W1;     db1 = dz1.sum(0)
        self.W1 -= lr * dW1; self.b1 -= lr * db1
        self.W2 -= lr * dW2; self.b2 -= lr * db2

In [ ]:
def cross_entropy(p, Y):
    return -np.mean(np.sum(Y * np.log(p + 1e-12), axis=1))

def accuracy(model, X, y):
    return np.mean(np.argmax(model.forward(X), axis=1) == y)

def train_mlp(model, X_tr, y_tr, X_te, y_te, n_epochs=100, batch_size=64,
              lr=0.5, weight_decay=0.0, rng=None):
    rng = rng or np.random.default_rng(0)
    Y_tr = one_hot(y_tr); n = len(X_tr)
    hist = {"loss": [], "train_acc": [], "test_acc": []}
    for ep in range(n_epochs):
        order = rng.permutation(n)
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            model.step(X_tr[idx], Y_tr[idx], lr, weight_decay)
        hist["loss"].append(cross_entropy(model.forward(X_tr), Y_tr))
        hist["train_acc"].append(accuracy(model, X_tr, y_tr))
        hist["test_acc"].append(accuracy(model, X_te, y_te))
    return hist

### Load the digits data

1,797 grayscale 8×8 images, flattened to 64-vectors, pixels scaled to $[0,1]$, random 80/20 split.

In [ ]:
digits = load_digits()
X_all = digits.data.astype(float) / 16.0
y_all = digits.target

n = len(X_all); perm = rng.permutation(n); n_tr = int(0.8 * n)
X_train, X_test = X_all[perm[:n_tr]], X_all[perm[n_tr:]]
y_train, y_test = y_all[perm[:n_tr]], y_all[perm[n_tr:]]
print(f"train: {X_train.shape}, test: {X_test.shape}")

## Part 1 — The randomization test

This is the heart of the paper. We train the **exact same network, the exact same way**, on two datasets:

1. **True labels** — the real digit labels.
2. **Random labels** — each training image is assigned a label drawn uniformly at random from $\{0,\dots,9\}$.
   The image–label relationship is now pure noise; there is *nothing to learn* and *nothing to generalize*.

If capacity control were the whole story, the network simply should not be able to drive the training error to
zero on the random-label set. Let's find out.

In [ ]:
y_random = rng.integers(0, 10, size=len(y_train))   # pure noise labels

net_true = MLP(64, 512, 10, np.random.default_rng(42))
hist_true = train_mlp(net_true, X_train, y_train, X_test, y_test,
                      n_epochs=120, lr=0.5, rng=np.random.default_rng(1))

net_rand = MLP(64, 512, 10, np.random.default_rng(42))
hist_rand = train_mlp(net_rand, X_train, y_random, X_test, y_test,
                      n_epochs=300, lr=0.5, rng=np.random.default_rng(1))

print(f"TRUE labels   -> train acc {hist_true['train_acc'][-1]*100:5.1f}%   test acc {hist_true['test_acc'][-1]*100:5.1f}%")
print(f"RANDOM labels -> train acc {hist_rand['train_acc'][-1]*100:5.1f}%   test acc {hist_rand['test_acc'][-1]*100:5.1f}%  (chance = 10%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_true['train_acc'], label='true labels')
axes[0].plot(hist_rand['train_acc'], label='random labels')
axes[0].axhline(1.0, ls=':', c='gray')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('TRAIN accuracy')
axes[0].set_title('Both reach ~100% training accuracy'); axes[0].legend()

axes[1].plot(hist_true['test_acc'], label='true labels')
axes[1].plot(hist_rand['test_acc'], label='random labels')
axes[1].axhline(0.1, ls=':', c='gray', label='chance (10%)')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('TEST accuracy')
axes[1].set_title('Only true labels generalize'); axes[1].legend()
plt.tight_layout(); plt.show()

**What just happened.** The network memorized the random labels — driving training accuracy to essentially 100% on a
dataset with *no signal whatsoever*. Its test accuracy on those random labels sits at chance (10%), because there is
nothing to generalize. On the true labels it reaches ~100% train **and** ~99% test.

Two conclusions, exactly as in the paper:

- **Effective capacity is enormous.** The network can fit $n$ arbitrary labels, so its effective VC dimension is
  $\ge n$. The classical bound $\hat R + O(\sqrt{d_{\mathrm{VC}}/n})$ is therefore vacuous: with $\hat R = 0$ it would
  permit *any* test error.
- **Generalization is not explained by the hypothesis class.** The *same* class, optimizer, and size generalize
  beautifully on real data and not at all on noise. The difference is in the **data**, not the capacity.

Notice also the *speed*: real labels converged in a handful of epochs; random labels took several times longer. Memorizing
noise is harder than learning structure — a hint we'll quantify next.

## Part 2 — The label-corruption sweep (Figure 1 of the paper)

Real data and pure noise are the two endpoints of a spectrum. Zhang et al. interpolate between them by corrupting a
**fraction $p$** of the labels: with probability $p$ a training label is replaced by a uniformly random one.
$p=0$ is the true dataset, $p=1$ is fully random.

For each $p$ we record three quantities, reproducing the three panels of the paper's Figure 1:
- **(a)** the training loss curve — does it still reach zero?
- **(b)** the *test* accuracy at convergence — how much generalization survives?
- **(c)** the *time to converge* (epochs to reach 99% train accuracy) — how much harder is memorization?

In [ ]:
def corrupt_labels(y, p, rng):
    y = y.copy()
    mask = rng.random(len(y)) < p
    y[mask] = rng.integers(0, 10, size=mask.sum())
    return y

fractions = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
results = []
for p in fractions:
    yc = corrupt_labels(y_train, p, np.random.default_rng(7))
    net = MLP(64, 512, 10, np.random.default_rng(42))
    h = train_mlp(net, X_train, yc, X_test, y_test,
                  n_epochs=300, lr=0.5, rng=np.random.default_rng(1))
    ta = np.array(h['train_acc'])
    epoch99 = int(np.argmax(ta >= 0.99)) if (ta >= 0.99).any() else len(ta)
    results.append({'p': p, 'hist': h, 'epoch99': epoch99})
    print(f"corrupt {p:.1f}: train {h['train_acc'][-1]*100:5.1f}%  test {h['test_acc'][-1]*100:5.1f}%  epochs->99%train: {epoch99}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) training loss curves still drive to ~0 for every corruption level
for r in results:
    axes[0].plot(r['hist']['loss'], label=f"p={r['p']:.1f}")
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('training loss')
axes[0].set_title('(a) Training loss -> 0 at every\ncorruption level'); axes[0].legend(fontsize=8)

# (b) generalization error grows with corruption
ps = [r['p'] for r in results]
test_err = [1 - r['hist']['test_acc'][-1] for r in results]
axes[1].plot(ps, test_err, 'o-')
axes[1].set_xlabel('fraction of corrupted labels'); axes[1].set_ylabel('test ERROR')
axes[1].set_title('(b) Generalization degrades\nsmoothly toward chance')

# (c) time to converge grows with corruption
axes[2].plot(ps, [r['epoch99'] for r in results], 'o-', c='darkgreen')
axes[2].set_xlabel('fraction of corrupted labels'); axes[2].set_ylabel('epochs to 99% train acc')
axes[2].set_title('(c) More noise = slower to memorize')
plt.tight_layout(); plt.show()

This reproduces the qualitative shape of the paper's Figure 1:

- **(a)** No matter how much noise, the optimizer still finds a global minimum with (near-)zero training loss. Capacity is
  never the bottleneck.
- **(b)** Test error rises *smoothly* from ~1% (clean) to ~90% (chance, fully random). With partly-noisy labels the network
  fits the true examples *and* memorizes the noisy ones, and generalization tracks the fraction of real signal.
- **(c)** The cleaner the labels, the faster training converges. Structure is easy to fit; noise must be memorized one point
  at a time. This is one of the few footholds for a capacity-free *explanation* of generalization (more in Part 4).

## Part 3 — Destroying the inputs too: random and Gaussian pixels

The paper goes further: it corrupts not just the labels but the **images themselves**, and the network still fits them.
We try two of their variants:

- **Shuffled pixels** — apply one fixed random permutation to the 64 pixels of every image (destroys spatial structure,
  keeps the pixel histogram).
- **Gaussian pixels** — replace every image with i.i.d. Gaussian noise matched to the data's mean and variance
  (no structure at all).

Labels are kept true here; the question is purely whether the network can still *fit* structureless inputs.

In [ ]:
# Shuffled pixels: one fixed permutation applied to all images
pix_perm = rng.permutation(64)
X_shuffled = X_train[:, pix_perm]

# Gaussian pixels: matched mean/std, otherwise pure noise
X_gauss = rng.standard_normal(X_train.shape) * X_train.std() + X_train.mean()

runs = {}
for name, Xin in [('true images', X_train), ('shuffled pixels', X_shuffled), ('gaussian pixels', X_gauss)]:
    net = MLP(64, 512, 10, np.random.default_rng(42))
    runs[name] = train_mlp(net, Xin, y_train, X_test, y_test,
                           n_epochs=200, lr=0.5, rng=np.random.default_rng(1))
    print(f"{name:16s}: final train acc {runs[name]['train_acc'][-1]*100:5.1f}%")

In [ ]:
plt.figure()
for name, h in runs.items():
    plt.plot(h['train_acc'], label=name)
plt.axhline(1.0, ls=':', c='gray')
plt.xlabel('epoch'); plt.ylabel('TRAIN accuracy')
plt.title('The network fits even pure-noise inputs to ~100% train accuracy')
plt.legend(); plt.show()

All three reach ~100% training accuracy. Whether the inputs are real digits, scrambled digits, or Gaussian static, the
network has the raw capacity to fit them. Interestingly, *more structured* inputs are again fit *faster* — the same
"easy first" effect we saw with labels.

## Part 4 — Why the VC bound goes vacuous, and finite-sample expressivity

### The bound, numerically

The VC generalization bound says, with high probability,
$$ R(f) \le \hat R(f) + O\!\left(\sqrt{\tfrac{d_{\mathrm{VC}}\log n}{n}}\right). $$
Parts 1–3 showed our network can shatter the training set: it fits $n \approx 1{,}400$ **arbitrary** labels, so
$d_{\mathrm{VC}} \ge n$. Plug that in:

In [ ]:
n_train = len(X_train)
d_vc = n_train          # we demonstrated the net shatters the training set, so VC >= n
bound = np.sqrt(d_vc * np.log(n_train) / n_train)
print(f"n = {n_train},  effective VC dim >= {d_vc}")
print(f"classical bound term  O(sqrt(VC*log n / n)) ~ {bound:.2f}")
print("=> the 'bound' on test error is larger than 1.0: it permits ANY error. Vacuous.")

### Finite-sample expressivity (the paper's Theorem 1)

The randomization experiments are *empirical*. The paper also proves the matching theory:

> **Theorem 1 (informal).** There exists a two-layer ReLU network with $2n + d$ parameters that can represent **any**
> function on a sample of size $n$ in $d$ dimensions.

So the ability to fit arbitrary labels is not an artifact of being "huge" — even a network barely larger than the dataset
can memorize it. Below we build the explicit construction in 1-D: given $n$ points $x_1 < x_2 < \dots < x_n$ on the line
and *any* target values $y_i$, a sum of $n$ ReLU "steps" reproduces all of them exactly. The weights are solved in closed
form, no gradient descent needed.

In [ ]:
def memorize_1d(x, y):
    """Build f(t) = sum_j a_j * relu(t - b_j) with f(x_i) = y_i exactly. Returns a, b."""
    order = np.argsort(x); x = x[order]; y = y[order]
    b = np.concatenate([[x[0] - 1.0], x[:-1]])     # kinks just left of each point
    relu = lambda t: np.maximum(0, t)
    # f(x_i) = sum_{j<=i} a_j (x_i - b_j); solve the triangular system point by point
    a = np.zeros(len(x))
    f_prev = 0.0
    for i in range(len(x)):
        # contribution of already-fixed kinks at x_i
        fixed = sum(a[j] * relu(x[i] - b[j]) for j in range(i))
        a[i] = (y[i] - fixed) / relu(x[i] - b[i])
    return a, b

def eval_1d(t, a, b):
    return np.array([np.sum(a * np.maximum(0, ti - b)) for ti in np.atleast_1d(t)])

x = np.sort(rng.uniform(0, 10, size=12))
y_targets = rng.uniform(-1, 1, size=12)        # arbitrary, structureless targets
a, b = memorize_1d(x, y_targets)

tt = np.linspace(x.min() - 0.5, x.max() + 0.5, 600)
plt.figure(figsize=(9, 4))
plt.plot(tt, eval_1d(tt, a, b), c='steelblue', label='2-layer ReLU net')
plt.scatter(x, y_targets, c='crimson', zorder=5, label='arbitrary targets')
plt.title('A 2-layer ReLU net with ~2n params fits any labeling of n points (Theorem 1)')
plt.xlabel('x'); plt.ylabel('f(x)'); plt.legend()
plt.show()
print('max fitting error:', np.abs(eval_1d(x, a, b) - y_targets).max())

The curve passes through every point exactly, for *any* targets we throw at it — the construction never fails. This is the
theoretical twin of the randomization test: **memorization is always within reach** for these models.

### So what *does* explain generalization?

If capacity does not, the explanation must involve the **interaction of the data, the architecture, and the optimizer**:

- **Implicit bias of SGD.** Among the many zero-training-error solutions, gradient descent tends to find "simple" ones
  (e.g. small-norm / large-margin), and reaches them faster when the data has structure (Parts 2c, 3).
- **Data-dependent measures.** Margins, norms, and compression-based bounds *can* be non-vacuous because they depend on the
  particular solution found, not on the worst case over the whole hypothesis class.
- **The right question.** Not "how large is the class?" but "which solution does training actually select, and why is it
  simple on real data?" Rethinking generalization, exactly as the title says.

## Exercises

### Exercise 9.0 — VC dimension by experiment

In Part 0 we checked shattering for one triangle and one square.

1. Generate 20 random sets of **3** points and 20 random sets of **4** points in the plane. Using `shatters`, estimate
   the fraction of each that a line can shatter. Confirm 3-point sets (in general position) are almost always shattered
   and 4-point sets never are.
2. The VC dimension of a linear classifier in $\mathbb{R}^d$ is $d+1$. Verify this for $d=1$ (a threshold on the line):
   show that 2 points are shattered but 3 are not. *(Hint: a 1-D linear classifier is $\mathrm{sign}(wx+b)$ — adapt
   `linearly_separable` to 1-D, or reason it out by hand.)*
3. Plug $d_{\mathrm{VC}} = d+1$ into the bound term $\sqrt{d_{\mathrm{VC}}\log n / n}$ for $n=1400$. Why is the bound
   *meaningful* for a linear model but *vacuous* for our MLP?

In [ ]:
# your code here

### Exercise 9.1 — Does regularization restore the classical story?

A natural hope: maybe **explicit regularization** prevents memorization of noise, rescuing capacity control. The paper
finds it does *not* — regularization is a useful tuning knob, not the explanation for generalization. Test this.

1. Re-run the **random-label** experiment from Part 1 with weight decay `weight_decay` $\in \{0, 10^{-4}, 10^{-3}, 10^{-2}\}$.
   Does the network still reach ~100% training accuracy on random labels for small/moderate weight decay?
2. Re-run the **true-label** experiment with the same settings and record test accuracy. Does regularization help
   generalization a little? A lot?
3. Discuss: based on your numbers, is regularization *necessary* for generalization, or merely *helpful*? Connect this to
   the paper's claim that "regularization may improve generalization but is neither necessary nor by itself sufficient."

In [ ]:
# your code here

### Exercise 9.2 — Capacity vs the train–test gap

Classical theory predicts that growing capacity should eventually *hurt* generalization (overfitting). Modern practice
often shows the opposite. Investigate on the true-label digits task.

1. Train the MLP with hidden widths $\{8, 16, 32, 64, 128, 256, 512, 1024\}$ (true labels, same other hyperparameters).
2. Plot final **train** and **test** accuracy vs hidden width (log-x). Where does the train–test gap stop growing — or
   does test accuracy keep *improving* with more parameters even past the point of zero training error?
3. For the two widest models, also fit **random** labels and record epochs-to-memorize. How does the capacity needed to
   memorize $n$ random labels relate to $n$? *(Recall Theorem 1: $\sim 2n$ parameters suffice.)*

In [ ]:
# your code here

## Takeaways

- **VC dimension** measures capacity as *"how many points can the class label arbitrarily?"* For a linear model it is $d+1$
  and the generalization bound $R \le \hat R + O(\sqrt{d_{\mathrm{VC}}/n})$ is meaningful.
- **Deep networks shatter their training set.** Zhang et al.'s randomization test — which we reproduced — shows a standard
  network fits **random labels** and even **random pixels** to zero training error. Its effective VC dimension is $\ge n$.
- **Therefore the classical bound is vacuous** for deep nets: it permits any test error. Capacity control alone cannot
  explain why they generalize.
- **The same model generalizes on real data and not on noise.** The difference lives in the *data* and in *which solution
  the optimizer selects*, not in the size of the hypothesis class.
- **Theorem 1** confirms the empirics: a 2-layer ReLU net with $\sim 2n$ parameters can fit any labeling of $n$ points.
- **Generalization in deep learning needs a rethink:** implicit bias of SGD, margins, and data-dependent (not worst-case)
  measures are where the real explanation lives.